# Version status: amplitude-analysis working copy

This is the local revised amplitude notebook, preserved for comparison with `plotroad_plusamplitudes.ipynb`; it is NOT certified as the final figure-producing version. Code is unchanged and outputs are cleared. It requires the August 27 waveform HDF5, phase picks, cable geometry and `mora-das-slope-100m.csv`; configure paths and confirm these inputs before running.

Important discrepancies: several plots still label strain as strain-rate, multiply noise RMS by source distance, or label a logarithmic axis as log10 of the quantity. These conflict with the revised figure discussed in the manuscript. Resolve the actual input units and intended panels before replacing the published notebook. Do not change waveform units based on plot labels alone.

The notebook is exploratory: variables are reused and some conversion operations modify arrays in place. Use a fresh kernel and review execution order; rerunning isolated cells may repeat a conversion. Saved figures are review outputs, not approved manuscript replacements.


## Load dependencies
Libraries support signal processing, coordinate transforms and plots. This step does not validate the physical units of the waveform.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyproj import CRS, Transformer
from matplotlib.lines import Line2D
from math import radians, cos, sin, atan2
import os
from scipy.signal import butter, filtfilt
from scipy.signal import hilbert
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import h5py
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.signal import butter, filtfilt
from matplotlib.ticker import LogLocator, NullFormatter
import seaborn as sns
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FixedLocator, FuncFormatter

## Read cable geometry and slope
Align channel identifiers across geometry, slope and pick tables before comparing amplitudes. Matching row positions alone is not a guarantee of channel correspondence.


In [ ]:
# Load the data

df1 = pd.read_csv('Paradise2NisquallyEntrace_Channels_interp_direction.csv')
dfslope = pd.read_csv('mora-das-slope-100m.csv')

# Drop rows with NaN values in Longitude or Latitude columns
df = df1.dropna(subset=['Longitude [°]', 'Latitude [°]']).reset_index(drop=True)

# Define the color mapping based on lithology
lithology_colors = {
    'Qs': '#FFD700',    # Gold
    'Qra': '#DC143C',   # Deep Pink
    'To': '#8B4513',     # Khaki
    'Tg': '#FF8C00',    # Dark Orange
    'Tdi': '#DC143C',   # Crimson
    'Td': '#9370DB',    # Medium Purple
}

# Define coordinate transformations
crs_wgs84 = CRS("EPSG:4326")     # WGS84 latitude and longitude
crs_utm10n = CRS("EPSG:32610")   # UTM zone 10N

# Create a transformer object
transformer = Transformer.from_crs(crs_wgs84, crs_utm10n, always_xy=True)

# Convert longitude and latitude to x and y in meters
x_meters, y_meters = transformer.transform(df['Longitude [°]'].values, df['Latitude [°]'].values)
df['x_meters'] = x_meters
df['y_meters'] = y_meters

# Convert meters to kilometers
df['x'] = df['x_meters'] / 1000
df['y'] = df['y_meters'] / 1000

# Earthquake coordinates (lat, lon) and convert to x, y in kilometers
quake_lat = 46.7722
quake_lon = -121.9588 
quake_x_meters, quake_y_meters = transformer.transform(quake_lon, quake_lat)
quake_x = quake_x_meters / 1000
quake_y = quake_y_meters / 1000

# Compute the midpoints between consecutive fiber points
lat_fibra = df['Latitude [°]'].values
lon_fibra = df['Longitude [°]'].values

lat_midpoints = (lat_fibra[1:] + lat_fibra[:-1]) / 2
lon_midpoints = (lon_fibra[1:] + lon_fibra[:-1]) / 2

# Convert midpoints to x, y in kilometers
x_mid_meters, y_mid_meters = transformer.transform(lon_midpoints, lat_midpoints)
x_mid = x_mid_meters / 1000
y_mid = y_mid_meters / 1000

# Compute azimuths and orientations
def compute_theta(x1, y1, x2, y2):
    return np.arctan2(y2 - y1, x2 - x1)

# Fiber orientation angles (theta_fiber)
theta_fiber = compute_theta(df['x'].values[:-1], df['y'].values[:-1], df['x'].values[1:], df['y'].values[1:])

# Direction from earthquake to midpoints (theta_quake)
theta_quake = compute_theta(np.full_like(x_mid, quake_x), np.full_like(y_mid, quake_y), x_mid, y_mid)

# Difference in angles
theta_diff = theta_quake - theta_fiber

# Ensure theta_diff is between -pi and pi
theta_diff = (theta_diff + np.pi) % (2 * np.pi) - np.pi

# Radial and tangential components
radial_component = np.cos(theta_diff) ** 2
tangential_component = np.sin(2 * theta_diff)

# Normalize components for plotting
radial_normalized = (radial_component - np.min(radial_component)) / (np.max(radial_component) - np.min(radial_component))
tangential_normalized = (tangential_component - np.min(tangential_component)) / (np.max(tangential_component) - np.min(tangential_component))


In [ ]:
df1

## Inspect the earthquake waveform
Load the configured event HDF5. The following filtered views help inspect arrivals; their display filters do not by themselves establish which samples feed the RMS calculation.


In [ ]:
# Parameters
chan_min = 0  # Starting channel index
chan_max = -1  # Use all channels
file = '/1-fnp/petasaur/p-jbod1/rainier/2023/08/27/decimator_2023-08-27_10.10.00_UTC.h5'

# Load data
with h5py.File(file, 'r') as rawdata:
    data = np.array(rawdata['Acquisition/Raw[0]/RawData'][:, chan_min:chan_max])
    #print(" Unidad de los datos:", data.attrs.get('Unit'))
    attrs= dict(rawdata['Acquisition'].attrs)
    time = np.array(rawdata['Acquisition/Raw[0]/RawDataTime'])
    dt = 1/attrs['MaximumFrequency']/2
    dx = attrs['SpatialSamplingInterval']
    nt,nx = data.shape

    x = np.linspace(0,nx*dx,nx)
    t = np.linspace(0,nt*dt,nt)
    
fs = 1/dt
b, a = butter(2, [1, 10], 'bandpass', fs=fs)
data_hf = filtfilt(b, a, data, axis=0)

In [ ]:
plt.subplots(dpi=300)
c=plt.imshow(data_hf.T,aspect='auto',cmap='seismic',vmin=-0.05, vmax=0.05)
plt.xlim(4000,6000)
plt.colorbar(c)
plt.show()

In [ ]:
chan_min =  0
chan_max = -1
low_cut1 =  1
# ======= CONFIG (edit) =======
file_path   = '/1-fnp/petasaur/p-jbod1/rainier/2023/08/27/decimator_2023-08-27_10.10.00_UTC.h5'
picks_path  = '/home/velgueta/notebooks/RainierDas/Notebooks/Phase-picking/pickerbyhour_2023-08-27_10/file_2023-08-27_10-10.csv'
zoom_start  = '2023-08-27 10:10:22'
zoom_end    = '2023-08-27 10:10:30'
chan_min    = 0
chan_max    = None        # None = all channels
low_cut, hi_cut = 1.0, 9.0

# ===== LOAD H5 =====
with h5py.File(file_path, 'r') as f:
    raw      = f['Acquisition/Raw[0]/RawData'][:, chan_min:chan_max]
    time_us  = np.array(f['Acquisition/Raw[0]/RawDataTime'])   # epoch microseconds
    attrs    = dict(f['Acquisition'].attrs)

fs    = float(attrs['MaximumFrequency'])*2.0
dx_km = float(attrs['SpatialSamplingInterval'])/1000.0

# ===== FILTER =====
b, a = butter(2, [low_cut/(fs/2.0), hi_cut/(fs/2.0)], btype='bandpass')
data_filt = filtfilt(b, a, raw.astype(float), axis=0)

# ===== TIME AXIS (seconds) & ZOOM =====
t_abs = pd.to_datetime(time_us.astype('int64'), unit='us', utc=True)
t0    = t_abs[0]
# Make 'ts' a NumPy array, not an Index
ts    = ((t_abs - t0) / pd.Timedelta(seconds=1)).to_numpy()

z0 = pd.to_datetime(zoom_start, utc=True)
z1 = pd.to_datetime(zoom_end,   utc=True)
z0s, z1s = (z0 - t0).total_seconds(), (z1 - t0).total_seconds()

# ===== PLOT IMAGE =====
fig, ax = plt.subplots(figsize=(14, 6))
ymax_km = data_filt.shape[1]*dx_km
ax.imshow(
    data_filt.T, cmap='seismic', aspect='auto',
    vmin=-0.1, vmax=0.1,
    extent=[ts[0], ts[-1], ymax_km, 0.0]
)
ax.set_xlim(max(ts[0], z0s), min(ts[-1], z1s))
ax.set_xlabel("Time (s)", fontsize=14)
ax.set_ylabel("Optical distance (km)", fontsize=14)
ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
ax.grid(alpha=0.2)

# ===== OVERLAY OFFICIAL P/S PICKS =====
dfp = pd.read_csv(picks_path)
for col in ('trace_p_arrival', 'trace_s_arrival'):
    if col in dfp.columns:
        dfp[col] = pd.to_datetime(dfp[col], utc=True, errors='coerce')
scode = pd.to_numeric(dfp.get('station_code', np.nan), errors='coerce')

p_done = s_done = False
for idx, row in dfp.iterrows():
    sc = scode.iloc[idx]
    if pd.isna(sc): 
        continue
    y_km = (int(sc) - int(chan_min)) * dx_km
    if y_km < 0 or y_km > ymax_km:
        continue

    tp = row.get('trace_p_arrival', pd.NaT)
    ts_ = row.get('trace_s_arrival', pd.NaT)

    if pd.notna(tp):
        tP = (tp - t0).total_seconds()
        ax.plot(tP, y_km, 'ro', ms=3, label='P-wave' if not p_done else "")
        p_done = True

    if pd.notna(ts_):
        tS = (ts_ - t0).total_seconds()
        ax.plot(tS, y_km, 'bo', ms=3, label='S-wave' if not s_done else "")
        s_done = True

if p_done or s_done:
    ax.legend(loc='upper right', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
plt.plot(dfp['trace_s_arrival'])
plt.plot(dfp['trace_p_arrival'])

In [ ]:
dfp

## Measure phase and background windows
The configured window is 1.5 seconds. Extract P/S windows from picks and a pre-P noise window; trace the use of raw arrays rather than assuming the display bandpass was applied. Missing/clipped samples are padded with NaN.


In [ ]:
noise_off= 10
win      = 1.5
chan_max = 3190
T, C     = raw.shape
nwin     = int(round(win*fs))
off      = int(round(noise_off*fs))
t_abs    = pd.to_datetime(time_us.astype('int64'),unit='us',utc=True)
t0       = t_abs[0]
# Step: Apply strain correction
# =============================================================================
gl       = 9.571428805203961  # Gauge Length in meters
correction_factor = (1550 * 1e-9) / (0.78 * 4 * np.pi * 1.4682 * gl)
print("Correction factor:", correction_factor)


dfp['trace_p_arrival'] = pd.to_datetime(dfp['trace_p_arrival'], utc=True, errors='coerce')
dfp['trace_s_arrival'] = pd.to_datetime(dfp['trace_s_arrival'], utc=True, errors='coerce')
dfp['station_code']    = pd.to_numeric(dfp['station_code'], errors='coerce')

# outputs to save the data
pwin = np.full((C,nwin),np.nan)
swin = np.full((C,nwin),np.nan)
noisewin = np.full((C,nwin),np.nan)

In [ ]:
# function to cut it
def put_window(dst, ch, start, length):
    """Copy raw[start:start+length, ch] into dst[ch, :], padding with NaN if clipped."""
    if ch < 0 or ch >= C: 
        return
    s = int(start); e = int(start + length)
    if e <= 0 or s >= T:
        return
    sclip, eclip = max(0, s), min(T, e)
    seg = raw[sclip:eclip, ch].astype(float)
    out = np.full(length, np.nan)
    left = sclip - s
    out[left:left+seg.size] = seg
    dst[ch, :] = out

In [ ]:
# --- Build windows from official picks (1 s each) ---
for _, r in dfp.iterrows():
    # channel (as a plain int)
    if pd.isna(r['station_code']):
        continue
    ch = int(r['station_code'])
    if not (0 <= ch < C):
        continue  # skip channels outside your sliced [0:3000)

    # ----- P window: [P, P + 1s)
    if pd.notna(r['trace_p_arrival']):
        p_idx = int(round((r['trace_p_arrival'] - t0).total_seconds() * fs))
        put_window(pwin, ch, p_idx, nwin)

        # noise: 1 s ending 5 s before P → [P - 5s - 1s, P - 5s)
        n_end = p_idx - off
        n_beg = n_end - nwin
        put_window(noisewin, ch, n_beg, nwin)

    # ----- S window: [S, S + 1s)
    if pd.notna(r['trace_s_arrival']):
        s_idx = int(round((r['trace_s_arrival'] - t0).total_seconds() * fs))
        put_window(swin, ch, s_idx, nwin)

print(f"fs = {fs:.2f} Hz | T={T}, C={C}, nwin={nwin} samples")
print("pwin:", pwin.shape, "swin:", swin.shape, "noisewin:", noisewin.shape)


## Convert and summarize amplitudes
Apply the configured phase-to-strain factor and calculate RMS. Verify instrument metadata first. In-place conversion means these cells should not be rerun on already converted arrays.


In [ ]:
SL = slice(0, 3191)           # usar canales 0–2999
R_floor_km = 0.5              # evitar 0 km
use_depth = True   
pwin = pwin*correction_factor
swin = swin*correction_factor
noisewin = noisewin*correction_factor 
# Masks: canal válido si tiene ventana
valid_noise = np.isfinite(noisewin[SL]).any(axis=1) 
valid_P = np.isfinite(pwin[SL]).any(axis=1)
valid_S = np.isfinite(swin[SL]).any(axis=1) 
valid_all = valid_noise & valid_P & valid_S # 
#Función RMS ignorando NaNs
def rms_nan(x):
    return np.sqrt(np.nanmean(x**2)) if np.isfinite(x).any() else np.nan

rms_P = np.array([rms_nan(row) for row in pwin[SL]]) 
rms_S = np.array([rms_nan(row) for row in swin[SL]]) 
rms_N = np.array([rms_nan(row) for row in noisewin[SL]])

SNR_P = rms_P / rms_N 
SNR_S = rms_S / rms_N # Mantener solo canales válidos 
idx = np.arange(3191) # eje x (0–2999)
idx_v = idx[valid_all] 
rms_Pv = rms_P[valid_all]
rms_Sv = rms_S[valid_all]
rms_Nv = rms_N[valid_all]
SNR_Pv = SNR_P[valid_all]
SNR_Sv = SNR_S[valid_all]

In [ ]:
# ===== PLOTS (prettier, same labels/titles) =====
def _moving_median(y, win=301):
    y = np.asarray(y, float)
    n = len(y)
    if n == 0 or win < 3: return y
    if win % 2 == 0: win += 1
    win = min(win, n if n % 2 == 1 else n-1)
    if win < 3: return y
    pad = win // 2
    ypad = np.pad(y, (pad, pad), mode='edge')
    out = np.empty_like(y, dtype=float)
    for i in range(n):
        out[i] = np.nanmedian(ypad[i:i+win])
    return out

def _pretty_log(ax):
    ax.set_yscale('log')
    ax.yaxis.set_major_locator(LogLocator(base=10.0, numticks=6))
    ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs=tuple(np.arange(2,10)*0.1), numticks=12))
    ax.yaxis.set_minor_formatter(NullFormatter())

# style
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300,
    "font.size": 12,
    "axes.grid": True, "grid.linestyle": "--", "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.labelpad": 8
})

# plotting copies so zeros don’t vanish on log axes
eps = 1e-15
rms_Nv_p = np.where(rms_Nv <= 0, eps, rms_Nv)
rms_Sv_p = np.where(rms_Sv <= 0, eps, rms_Sv)
rms_Pv_p = np.where(rms_Pv <= 0, eps, rms_Pv)
SNR_Pv_p = np.where(SNR_Pv <= 0, eps, SNR_Pv)
SNR_Sv_p = np.where(SNR_Sv <= 0, eps, SNR_Sv)

# subtle trend lines (moving medians)
win = min(301, max(3, 2*len(idx_v)-1))  # auto-adjust if few points
N_med = _moving_median(rms_Nv_p, win)
S_med = _moving_median(rms_Sv_p, win)
P_med = _moving_median(rms_Pv_p, win)
SNRP_med = _moving_median(SNR_Pv_p, win)
SNRS_med = _moving_median(SNR_Sv_p, win)

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
dx=9.7
# Panel 1: S vs Noise (keep your text exactly)
ax = axes[0]
sN = ax.scatter(idx_v*dx, rms_Nv_p, s=10, c='gray', alpha=0.4, rasterized=True, label='Noise RMS')
sS = ax.scatter(idx_v*dx, rms_Sv_p, s=10, c='blue', alpha=0.7, rasterized=True, label='S RMS')
ax.plot(idx_v*dx, N_med, lw=2.0, alpha=0.95, color='gray')
ax.plot(idx_v*dx, S_med, lw=2.0, alpha=0.95, color='blue')
ax.set_ylabel('Amplitude (/s)')
#ax.set_title('RMS amplitudes per channel (0–3000)')
_pretty_log(ax)
ax.grid(alpha=0.3, ls='--')
ax.legend(frameon=False)

# Panel 2: P vs Noise (keep your text exactly)
ax = axes[1]
ax.scatter(idx_v*dx, rms_Nv_p, s=10, c='gray', alpha=0.4, rasterized=True, label='Noise RMS')
ax.scatter(idx_v*dx, rms_Pv_p, s=10, c='red',  alpha=0.7, rasterized=True, label='P RMS')
ax.plot(idx_v*dx, N_med, lw=2.0, alpha=0.95, color='gray')
ax.plot(idx_v*dx, P_med, lw=2.0, alpha=0.95, color='red')
ax.set_ylabel('Amplitude (/s)')
#ax.set_title('RMS amplitudes per channel (0–3000)')
_pretty_log(ax)
ax.grid(alpha=0.3, ls='--')
ax.legend(frameon=False)

# Panel 3: SNR P/S and noise rms (keep exactly what you had)
ax = axes[2]
sP = ax.scatter(idx_v*dx, SNR_Pv_p, s=10, c='red',  alpha=0.7, rasterized=True, label='SNR P')
sS = ax.scatter(idx_v*dx, SNR_Sv_p, s=10, c='blue', alpha=0.7, rasterized=True, label='SNR S')
ax.plot(idx_v*dx, SNRP_med, lw=2.0, alpha=0.95, color='red')
ax.plot(idx_v*dx, SNRS_med, lw=2.0, alpha=0.95, color='blue')
# you plotted noise rms here too; kept as-is
ax.scatter(idx_v*dx, rms_Nv_p, s=10, c='gray', alpha=0.7, rasterized=True, label='Noise rms')
ax.set_ylabel('SNR (linear)')
ax.set_xlabel('Channel')
_pretty_log(ax)
ax.grid(alpha=0.3, ls='--')
ax.legend(frameon=False)

plt.tight_layout()
plt.savefig('rms_amplitude.pdf',dpi=300)
plt.show()


## Account for source distance
Compute the source-channel distance and multiply signal RMS by R to explore geometrical spreading. This produces strain times km when R is in km, not strain-rate. Source-distance correction is not physically appropriate for ambient noise.


In [ ]:
use_depth = True              # si tienes ev_depth_km, úsalo
# ==========================

# -------- Helpers: distancia hipócentral --------
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1 = np.radians(lat1); lon1 = np.radians(lon1)
    lat2 = np.radians(lat2); lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

def compute_dist_km(chan_lat, chan_lon, ev_lat, ev_lon, ev_depth_km=0.0, chan_elev_km=None):
    chan_lat = np.asarray(chan_lat, float)
    chan_lon = np.asarray(chan_lon, float)
    epi = haversine_km(chan_lat, chan_lon, ev_lat, ev_lon)
    if chan_elev_km is not None:
        z = np.asarray(chan_elev_km, float)
        z = np.clip(ev_depth_km - z, 0.0, None)
    else:
        z = np.full_like(epi, float(ev_depth_km))
    return np.sqrt(epi**2 + z**2)

# --------- 1) Selección de datos (0–3000) ---------
idx = np.arange(pwin.shape[0])[SL]

def rms_nan(row):
    return np.sqrt(np.nanmean(row**2)) if np.isfinite(row).any() else np.nan

valid = np.isfinite(rms_P) & np.isfinite(rms_S) & np.isfinite(rms_N)

# --------- 2) Distancia R (elige mejor fuente disponible) ---------
C_sel = len(idx)

dist_km = None

# ========= EVENT INFO =========
ev_lat = 46.7722
ev_lon = -121.9588
ev_depth_km = 9.5   # depth in km

# ========= DISTANCE (0–3000 channels) =========
# Try to get lat/lon per channel
if ('df1' in globals() and "Latitude [°]" in df1.columns and "Longitude [°]" in df1.columns):
    chan_lat = df1["Latitude [°]"].to_numpy(dtype=float)[SL]
    chan_lon = df1["Longitude [°]"].to_numpy(dtype=float)[SL]

    # use depth-corrected distance
    dist_km = compute_dist_km(
        chan_lat, chan_lon,
        ev_lat, ev_lon,
        ev_depth_km,
        globals().get('chan_elev_km', None)
    )

# If you don't have lat/lon, fallback to fiber distance
elif 'optical_dist_m' in globals():
    dist_km = np.asarray(optical_dist_m, float)[SL] / 1000.0

elif 'dx_m' in globals():
    dist_km = (np.arange(pwin.shape[0], dtype=float)[SL] * float(dx_m)) / 1000.0

else:
    print("WARNING: No coords — using channel index as distance proxy (not physical)")
    dist_km = np.arange(len(rms_P), dtype=float) + 1

# Ensure no zeros
R_used = np.where(np.isfinite(dist_km) & (dist_km>0.5), dist_km, 0.5)

In [ ]:
min(R_used)

In [ ]:
# --- build corrected amplitudes (×R) for all selected channels ---
AmpP_corr_all = rms_P * R_used
AmpS_corr_all = rms_S * R_used
AmpN_corr_all = rms_N * R_used

# --- validity mask so x and y have same length ---
valid_mask = np.isfinite(rms_P) & np.isfinite(rms_S) & np.isfinite(rms_N) & np.isfinite(R_used)

# --- x-axis: channel indices 0..2999 (after SL = slice(0,3000)) ---
ch = np.arange(len(rms_P), dtype=int)
ch_v = ch[valid_mask]

# --- masked series ---
P_raw_v  = rms_P[valid_mask]
S_raw_v  = rms_S[valid_mask]
N_raw_v  = rms_N[valid_mask]
P_corr_v = AmpP_corr_all[valid_mask]
S_corr_v = AmpS_corr_all[valid_mask]
N_corr_v = AmpN_corr_all[valid_mask]

# ---- plots: raw and corrected vs channel ----
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# (1) raw RMS amplitudes vs channel
#axes[0].scatter(ch_v, N_raw_v, s=10, c='gray', alpha=0.35, label='Noise RMS')
#axes[0].scatter(ch_v, P_raw_v, s=10, c='red',  alpha=0.7,  label='P RMS')
#axes[0].scatter(ch_v, S_raw_v, s=10, c='blue', alpha=0.7,  label='S RMS')
#axes[0].set_ylabel('RMS amplitude (raw)')
#axes[0].set_title('Raw amplitudes vs channel (0–2999)')
#axes[0].grid(ls='--', alpha=0.3)
#axes[0].legend()

# (2) 1/R-corrected amplitudes vs channel
#axes[1].scatter(ch_v, N_corr_v, s=10, c='gray', alpha=0.35, label='Noise RMS × R')
#axes[1].scatter(ch_v, P_corr_v, s=10, c='red',  alpha=0.7,  label='P RMS × R')
#axes[1].scatter(ch_v, S_corr_v, s=10, c='blue', alpha=0.7,  label='S RMS × R')
#axes[1].set_xlabel('Channel')
#axes[1].set_ylabel('RMS amplitude × R')
#axes[1].set_title('1/R-corrected amplitudes vs channel (0–2999)')
#axes[1].grid(ls='--', alpha=0.3)
#axes[1].legend()

#plt.tight_layout()
#plt.show()

# --- quick summary ---
#print(f"Plotted channels (valid): {valid_mask.sum()} / {len(rms_P)}")
#print("Median RAW   RMS  P / S / N:", np.nanmedian(P_raw_v), np.nanmedian(S_raw_v), np.nanmedian(N_raw_v))
#print("Median CORR  RMS  P / S / N:", np.nanmedian(P_corr_v), np.nanmedian(S_corr_v), np.nanmedian(N_corr_v))


In [ ]:
def plot_psn_pretty_km_from_channel(
    rms_P, rms_S, rms_N, R_used,
    meters_per_channel=9.7,
    chan_slice=None,         # e.g., (0, 3000)
    zero_at_slice=True,      # x=0 at slice start if True
    point_size=9,
    smooth_win=301
):
    # --- helpers (visual only) ---
    def _moving_median(y, win=301):
        y = np.asarray(y, float)
        n = len(y)
        if n == 0 or win < 3: return y
        if win % 2 == 0: win += 1
        win = min(win, n if n % 2 == 1 else n-1)
        if win < 3: return y
        pad = win // 2
        ypad = np.pad(y, (pad, pad), mode='edge')
        out = np.empty_like(y, dtype=float)
        for i in range(n):
            out[i] = np.nanmedian(ypad[i:i+win])
        return out

    def _nice_log(ax):
        ax.set_yscale("log")
        ax.yaxis.set_major_locator(LogLocator(base=10.0, numticks=6))
        ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs=tuple(np.arange(2,10)*0.1), numticks=12))
        ax.yaxis.set_minor_formatter(NullFormatter())

    # --- inputs as arrays (everything already measured) ---
    rms_P = np.asarray(rms_P, float)
    rms_S = np.asarray(rms_S, float)
    rms_N = np.asarray(rms_N, float)
    R_used = np.asarray(R_used, float)

    # ×R correction (exactly your definition)
    P_cor_all = rms_P * R_used
    S_cor_all = rms_S * R_used
    N_cor_all = rms_N * R_used

    # validity + (optional) slice
    valid = np.isfinite(rms_P) & np.isfinite(rms_S) & np.isfinite(rms_N) & np.isfinite(R_used)
    ch = np.arange(len(rms_P), dtype=int)
    if chan_slice is None:
        s0, s1 = 0, len(ch)
    else:
        s0, s1 = chan_slice
    use = valid & (ch >= s0) & (ch < s1)
    if use.sum() == 0:
        raise RuntimeError("No valid data to plot (check chan_slice and NaNs/inf).")

    # X-axis = distance in km from channel index (1 ch = meters_per_channel)
    dist_km_abs = ch[use] * meters_per_channel / 1000.0
    x = dist_km_abs - (s0 * meters_per_channel / 1000.0) if zero_at_slice else dist_km_abs

    # vectors (masked)
    P_raw, S_raw, N_raw = rms_P[use], rms_S[use], rms_N[use]
    P_cor, S_cor, N_cor = P_cor_all[use], S_cor_all[use], N_cor_all[use]

    # tiny epsilon so zeros don’t vanish on log (plotting copy only)
    eps = 1e-15
    P_raw_p = np.where(P_raw <= 0, eps, P_raw)
    S_raw_p = np.where(S_raw <= 0, eps, S_raw)
    N_raw_p = np.where(N_raw <= 0, eps, N_raw)
    P_cor_p = np.where(P_cor <= 0, eps, P_cor)
    S_cor_p = np.where(S_cor <= 0, eps, S_cor)
    N_cor_p = np.where(N_cor <= 0, eps, N_cor)

    # smooth trends (auto-size window if slice is short)
    sw = min(smooth_win if smooth_win % 2 else smooth_win+1, max(3, 2*len(x)-1))
    P_raw_med = _moving_median(P_raw_p, sw)
    S_raw_med = _moving_median(S_raw_p, sw)
    N_raw_med = _moving_median(N_raw_p, sw)
    P_cor_med = _moving_median(P_cor_p, sw)
    S_cor_med = _moving_median(S_cor_p, sw)
    N_cor_med = _moving_median(N_cor_p, sw)

    # --- aesthetics (professional, minimal) ---
    plt.rcParams.update({
        "figure.dpi": 120, "savefig.dpi": 300,
        "font.size": 12,
        "axes.grid": True, "grid.linestyle": "--", "grid.alpha": 0.25,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.labelpad": 8
    })
    # consistent, print-safe colors
    colP, colS, colN = "#d62728", "#1f77b4", "#7f7f7f"  # red, blue, gray

    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    # --- RAW (s^-1) ---
    ax = axes[0]
    sN = ax.scatter(x, N_raw_p, s=point_size, c=colN, alpha=0.55, rasterized=True)
    sP = ax.scatter(x, P_raw_p, s=point_size, c=colP, alpha=0.85, rasterized=True)
    sS = ax.scatter(x, S_raw_p, s=point_size, c=colS, alpha=0.85, rasterized=True)
    if len(x) > 1:
        ax.plot(x, N_raw_med, lw=2.0, alpha=0.95, color=colN)
        ax.plot(x, P_raw_med, lw=2.0, alpha=0.95, color=colP)
        ax.plot(x, S_raw_med, lw=2.0, alpha=0.95, color=colS)
    _nice_log(ax)
    ax.set_ylabel(r'Amplitude (strain-rate, $s^{-1}$)')
    ax.legend([sP, sS, sN], ["P wave", "S wave ", "Noise level"], loc="upper left", frameon=False, handletextpad=0.6)

    # --- ×R (s^-1·m) ---
    ax = axes[1]
    ax.scatter(x, N_cor_p, s=point_size, c=colN, alpha=0.55, rasterized=True)
    ax.scatter(x, P_cor_p, s=point_size, c=colP, alpha=0.80, rasterized=True)
    ax.scatter(x, S_cor_p, s=point_size, c=colS, alpha=0.80, rasterized=True)
    if len(x) > 1:
        ax.plot(x, N_cor_med, lw=2.0, alpha=0.90, color=colN)
        ax.plot(x, P_cor_med, lw=2.0, alpha=0.90, color=colP)
        ax.plot(x, S_cor_med, lw=2.0, alpha=0.90, color=colS)
    _nice_log(ax)
    ax.set_xlabel("Distance along fiber (km)")  # keep or remove as you prefer
    ax.set_ylabel(r'Amplitude x R(strain-rate, $s^{-1}$)')

    plt.tight_layout()
    plt.savefig('rms_plot2.pdf',dpi=300)
    plt.show()
    
plot_psn_pretty_km_from_channel(
    rms_P, rms_S, rms_N, R_used,
    meters_per_channel=9.7,          # or 9.7
    chan_slice=(0, len(rms_P)),      # safe default slice
    zero_at_slice=True,
    point_size=9,
    smooth_win=301
)


In [ ]:
def plot_snr_ps_km(
    rms_P, rms_S, rms_N,
    meters_per_channel=9.7,   # set 9.6 if that's your spacing
    chan_slice=None,          # None, (start, stop), or int (stop)
    zero_at_slice=True,       # x=0 at slice start
    point_size=9,
    smooth_win=301,           # odd number
    show_noise=True           # include Noise RMS panel
):
    # --- helpers (visual only) ---
    def _moving_median(y, win=301):
        y = np.asarray(y, float)
        n = len(y)
        if n == 0 or win < 3: return y
        if win % 2 == 0: win += 1
        win = min(win, n if n % 2 == 1 else n-1)
        if win < 3: return y
        pad = win // 2
        ypad = np.pad(y, (pad, pad), mode='edge')
        out = np.empty_like(y, dtype=float)
        for i in range(n):
            out[i] = np.nanmedian(ypad[i:i+win])
        return out

    def _nice_log(ax):
        ax.set_yscale("log")
        ax.yaxis.set_major_locator(LogLocator(base=10.0, numticks=6))
        ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs=tuple(np.arange(2,10)*0.1), numticks=12))
        ax.yaxis.set_minor_formatter(NullFormatter())

    # --- arrays ---
    rms_P = np.asarray(rms_P, float)
    rms_S = np.asarray(rms_S, float)
    rms_N = np.asarray(rms_N, float)

    if not (len(rms_P) == len(rms_S) == len(rms_N)):
        raise ValueError("Input arrays must have the same length.")

    ch = np.arange(len(rms_P), dtype=int)
    if chan_slice is None:
        s0, s1 = 0, len(ch)
    elif isinstance(chan_slice, int):
        s0, s1 = 0, chan_slice
    else:
        s0, s1 = chan_slice
    s0 = max(0, int(s0)); s1 = min(len(ch), int(s1))

    valid = np.isfinite(rms_P) & np.isfinite(rms_S) & np.isfinite(rms_N) & (rms_N > 0)
    use = valid & (ch >= s0) & (ch < s1)
    if use.sum() == 0:
        raise RuntimeError("No valid data to plot (check chan_slice and NaNs/zeros in rms_N).")

    # x in km from channel index
    dist_km_abs = ch[use] * meters_per_channel / 1000.0
    x = dist_km_abs - (s0 * meters_per_channel / 1000.0) if zero_at_slice else dist_km_abs

    # SNR (dimensionless)
    eps = 1e-12
    N = rms_N[use]
    snrP = rms_P[use] / (N + eps)
    snrS = rms_S[use] / (N + eps)

    # smooth trends (auto-size window if slice is short)
    sw = min(smooth_win if smooth_win % 2 else smooth_win+1, max(3, 2*len(x)-1))
    snrP_med = _moving_median(snrP, sw)
    snrS_med = _moving_median(snrS, sw)
    noise_med = _moving_median(N, sw)

    # --- aesthetics (professional, minimal) ---
    plt.rcParams.update({
        "figure.dpi": 120, "savefig.dpi": 300,
        "font.size": 12,
        "axes.grid": True, "grid.linestyle": "--", "grid.alpha": 0.25,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.labelpad": 8
    })
    colP, colS, colN = "#d62728", "#1f77b4", "#7f7f7f"  # red, blue, gray

    rows = 1 #+ int(show_noise)
    fig, axes = plt.subplots(rows, 1, figsize=(12, 7 + 2*(rows-1)), sharex=True)
    if rows == 1:
        axes = np.array([axes])

    r = 0

    # (optional) Noise RMS panel
    #if show_noise:
    #    ax = axes[r]; r += 1
    #    ax.scatter(x, N, s=point_size, c=colN, alpha=0.7, rasterized=True)
    #    if len(x) > 1:
    #        ax.plot(x, noise_med, lw=2.0, alpha=0.95, color=colN)
    #    _nice_log(ax)
    #    ax.set_ylabel(r"Noise RMS ($s^{-1}$)")

    # Linear SNR panel (log y)
    ax = axes[r]; r =+1
    sP = ax.scatter(x, snrP, s=point_size, c=colP, alpha=0.85, rasterized=True)
    sS = ax.scatter(x, snrS, s=point_size, c=colS, alpha=0.85, rasterized=True)
    sS = ax.scatter(x, N/correction_factor, s=point_size, c=colN, alpha=0.85, rasterized=True)
    if len(x) > 1:
        ax.plot(x, snrP_med, lw=2.0, alpha=0.5, color=colP)
        ax.plot(x, snrS_med, lw=2.0, alpha=0.5, color=colS)
    _nice_log(ax)
    ax.set_ylabel("Noise rms/ SNR ")
    #ax.legend([sP, sS], ["P", "S"], loc="upper left", frameon=False, handletextpad=0.6)

    # shared x label
    axes[-1].set_xlabel("Distance along fiber (km)")

    plt.tight_layout()
    plt.savefig('snr_plot.pdf',dpi=300)
    plt.show()

# Example (dB panel removed):
plot_snr_ps_km(
    rms_P, rms_S, rms_N,
    meters_per_channel=9.7,   # set your spacing
    chan_slice=(0, 3190),
    zero_at_slice=True,
    point_size=9,
    smooth_win=301,
    show_noise=True
)


## Compare amplitudes with slope
Plot observations and equal-width slope bins with median/bootstrap summaries. The later call uses 20 bins; determine the actual width from bin edges rather than assuming a universal width.


In [ ]:
def plot_rms_vs_slope(
    dfslope,
    rms_P, rms_S,
    chan_slice=None,
    point_size=12,
    smooth_bins=18,
    n_boot=200,
    title=None,
):
    """
    Pretty RMS (P=red, S=blue) vs slope (degrees), per channel.
    - dfslope: DataFrame with columns 'Channel' and 'slope_deg'
    - rms_P, rms_S: arrays aligned to dfslope rows
    - chan_slice: None or (start, stop) channel-index range [start, stop)
    - smooth_bins: number of equal-width slope bins for median smoothing
    - n_boot: bootstrap draws for 95% CI (set 0 to disable)
    """

    # Theme to match previous "pro" figures
    try:
        import seaborn as sns
        sns.set_theme(style="whitegrid", context="talk")
    except Exception:
        pass
    plt.rcParams.update({
        "figure.dpi": 140, "savefig.dpi": 300,
        "axes.linewidth": 1.2,
        "axes.spines.top": False, "axes.spines.right": False,
        "grid.color": "0.85", "grid.linestyle": "--", "grid.linewidth": 0.8,
        "axes.labelpad": 9, "xtick.major.pad": 6, "ytick.major.pad": 6,
    })

    # ---- align + select ----
    ch    = dfslope["Channel"].to_numpy()
    slope = dfslope["slope_deg"].to_numpy(dtype=float)
    rms_P = np.asarray(rms_P, float)
    rms_S = np.asarray(rms_S, float)

    if chan_slice is None:
        s0, s1 = np.nanmin(ch), np.nanmax(ch) + 1
    else:
        s0, s1 = chan_slice

    use_slice = (ch >= s0) & (ch < s1)
    valid = np.isfinite(slope) & np.isfinite(rms_P) & np.isfinite(rms_S)
    use = valid & use_slice
    if use.sum() == 0:
        raise RuntimeError("No valid points in the requested range (check NaNs and chan_slice).")

    x   = slope[use]
    yP  = rms_P[use]
    yS  = rms_S[use]

    # Avoid log(0): keep original arrays intact; clip only for plotting/stats
    eps = 1e-15
    yPp = np.maximum(yP, eps)
    ySp = np.maximum(yS, eps)

    # ---- binned medians over slope ----
    xmin, xmax = float(np.nanmin(x)), float(np.nanmax(x))
    nb = max(3, int(smooth_bins))
    edges = np.linspace(xmin, xmax, nb + 1)
    xc = 0.5 * (edges[:-1] + edges[1:])

    def bin_collect(xv, yv):
        meds = np.full(nb, np.nan)
        masks = []
        for i in range(nb):
            m = (xv >= edges[i]) & (xv < edges[i+1])
            masks.append(m)
            if m.any():
                meds[i] = np.nanmedian(yv[m])
        return meds, masks

    medP, masksP = bin_collect(x, yPp)
    medS, masksS = bin_collect(x, ySp)

    # ---- bootstrap 95% CI for bin medians (optional) ----
    rng = np.random.default_rng(42)
    def boot_ci(yv, masks, nboot=200, alpha=0.05):
        lo = np.full(nb, np.nan)
        hi = np.full(nb, np.nan)
        if nboot <= 0:
            return lo, hi
        for i, m in enumerate(masks):
            if not m.any():
                continue
            vals = np.asarray(yv[m], float)
            if len(vals) < 3:
                med = np.nanmedian(vals)
                lo[i] = hi[i] = med
                continue
            idx = rng.integers(0, len(vals), size=(nboot, len(vals)))
            boots = np.median(vals[idx], axis=1)
            lo[i] = np.percentile(boots, 100*alpha/2)
            hi[i] = np.percentile(boots, 100*(1 - alpha/2))
        return lo, hi

    loP, hiP = boot_ci(yPp, masksP, nboot=n_boot)
    loS, hiS = boot_ci(ySp, masksS, nboot=n_boot)

    # ---- colors (consistent with SNR plot) ----
    colP, colS = "#d62728", "#1f77b4"  # red P, blue S

    # ---- figure ----
    fig, ax = plt.subplots(figsize=(12.5, 6.0))

    # CI bands
    if n_boot and np.any(np.isfinite(loP)):
        ax.fill_between(xc, loP, hiP, color=colP, alpha=0.12, zorder=1, label="_nolegend_")
    if n_boot and np.any(np.isfinite(loS)):
        ax.fill_between(xc, loS, hiS, color=colS, alpha=0.12, zorder=1, label="_nolegend_")

    # Median lines
    ax.plot(xc, medP, color=colP, lw=2.6, alpha=0.95, zorder=3, label="Median P (binned)")
    ax.plot(xc, medS, color=colS, lw=2.6, alpha=0.95, zorder=3, label="Median S (binned)")

    # Scatter observations
    ax.scatter(x, yPp, s=point_size, c=colP, alpha=0.35, edgecolors="none",
               rasterized=True, label="P observations", zorder=2)
    ax.scatter(x, ySp, s=point_size, c=colS, alpha=0.35, edgecolors="none",
               rasterized=True, label="S observations", zorder=2)

    # Log y with clean ticks
    ax.set_yscale("log")
    ax.yaxis.set_major_locator(LogLocator(base=10.0, numticks=6))
    ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs=tuple(range(2, 10)), numticks=12))
    ax.yaxis.set_minor_formatter(NullFormatter())

    # Labels / limits
    pad = 0.02 * (xmax - xmin if xmax > xmin else 1.0)
    ax.set_xlim(xmin - pad, xmax + pad)
    ax.set_xlabel("Slope (degrees)")
    ax.set_ylabel(r"RMS amplitude (strain-rate, $s^{-1}$)")
    if title:
        ax.set_title(title, pad=10)

    # Legend
    handles = [
        Line2D([0], [0], marker='o', color='none', markerfacecolor=colP, alpha=0.55,
               markersize=7, label="P observations"),
        Line2D([0], [0], marker='o', color='none', markerfacecolor=colS, alpha=0.55,
               markersize=7, label="S observations"),
        Line2D([0], [0], color=colP, lw=2.6, label="Median P (binned)"),
        Line2D([0], [0], color=colS, lw=2.6, label="Median S (binned)"),
    ]
    if n_boot:
        handles.append(Patch(facecolor="0.3", alpha=0.12, edgecolor="none", label="95% CI (bootstrap)"))
    ax.legend(handles=handles, loc="lower right", frameon=True, framealpha=0.1)

    ax.grid(True, which="major", linestyle="--", linewidth=0.8, alpha=0.35)
    fig.tight_layout()
    return fig, ax


In [ ]:
def plot_snr_vs_slope(
    dfslope,
    rms_P, rms_S, rms_N,
    chan_slice=None,
    point_size=10,
    smooth_bins=18,
    n_boot=200,
    title=None,
):
    """
    Pretty SNR (P and S) vs slope (degrees), with binned medians and 95% CI.

    Parameters
    ----------
    dfslope : DataFrame
        Must have columns 'Channel' and 'slope_deg'.
    rms_P, rms_S, rms_N : array-like
        Same length/order as dfslope rows. SNR = rms_signal / rms_noise.
    chan_slice : tuple (start, stop) or None
        Channel-index range to include: [start, stop).
    point_size : int
        Marker size for scatter points.
    smooth_bins : int
        Number of equal-width bins in slope for median smoothing.
    n_boot : int
        Bootstrap draws (per bin) for 95% CI on medians. Set 0 to disable CI.
    title : str or None
        Optional axis title.
    """
    # Theme (matches the violin plot style)
    try:
        import seaborn as sns
        sns.set_theme(style="whitegrid", context="talk")
    except Exception:
        pass
    plt.rcParams.update({
        "figure.dpi": 140, "savefig.dpi": 300,
        "axes.linewidth": 1.2,
        "axes.spines.top": False, "axes.spines.right": False,
        "grid.color": "0.85", "grid.linestyle": "--", "grid.linewidth": 0.8,
        "axes.labelpad": 9, "xtick.major.pad": 6, "ytick.major.pad": 6,
    })

    # ---- align + select ----
    ch    = dfslope["Channel"].to_numpy()
    slope = dfslope["slope_deg"].to_numpy(dtype=float)
    rms_P = np.asarray(rms_P, float)
    rms_S = np.asarray(rms_S, float)
    rms_N = np.asarray(rms_N, float)

    if chan_slice is None:
        s0, s1 = np.nanmin(ch), np.nanmax(ch) + 1
    else:
        s0, s1 = chan_slice

    use_slice = (ch >= s0) & (ch < s1)
    valid = (np.isfinite(slope) & np.isfinite(rms_P) &
             np.isfinite(rms_S) & np.isfinite(rms_N) & (rms_N > 0))
    use = valid & use_slice
    if use.sum() == 0:
        raise RuntimeError("No valid points for SNR vs slope (check NaNs/zeros and chan_slice).")

    x = slope[use]
    eps = 1e-12
    snrP = rms_P[use] / (rms_N[use] + eps)
    snrS = rms_S[use] / (rms_N[use] + eps)

    # ---- binned medians (smooth trend over slope) ----
    xmin, xmax = float(np.nanmin(x)), float(np.nanmax(x))
    nb = max(3, int(smooth_bins))
    edges = np.linspace(xmin, xmax, nb + 1)
    xc = 0.5 * (edges[:-1] + edges[1:])

    def bin_stat(y, fn=np.nanmedian):
        med = np.full(nb, np.nan)
        masks = []
        for i in range(nb):
            m = (x >= edges[i]) & (x < edges[i+1])
            masks.append(m)
            if m.any():
                med[i] = fn(y[m])
        return med, masks

    medP, masksP = bin_stat(snrP)
    medS, masksS = bin_stat(snrS)

    # Bootstrap CI on median per bin (optional)
    rng = np.random.default_rng(42)
    def boot_ci(y, masks, alpha=0.05, nboot=200):
        lo = np.full(nb, np.nan)
        hi = np.full(nb, np.nan)
        if nboot <= 0:
            return lo, hi
        for i, m in enumerate(masks):
            if not m.any():
                continue
            vals = np.asarray(y[m], float)
            if len(vals) < 3:
                # too few points → CI collapses to median
                med = np.nanmedian(vals)
                lo[i] = hi[i] = med
                continue
            idx = rng.integers(0, len(vals), size=(nboot, len(vals)))
            boots = np.median(vals[idx], axis=1)
            lo[i] = np.percentile(boots, 100*alpha/2)
            hi[i] = np.percentile(boots, 100*(1 - alpha/2))
        return lo, hi

    loP, hiP = boot_ci(snrP, masksP, nboot=n_boot)
    loS, hiS = boot_ci(snrS, masksS, nboot=n_boot)

    # ---- colors (consistent, readable) ----
    colP, colS = "#d62728", "#1f77b4"  # red for P, blue for S

    # ---- figure ----
    fig, ax = plt.subplots(figsize=(12.5, 6.0))

    # CI bands (draw first, under everything)
    if n_boot and np.any(np.isfinite(loP)):
        ax.fill_between(xc, loP, hiP, color=colP, alpha=0.12, zorder=1, label="_nolegend_")
    if n_boot and np.any(np.isfinite(loS)):
        ax.fill_between(xc, loS, hiS, color=colS, alpha=0.12, zorder=1, label="_nolegend_")

    # Median lines
    ax.plot(xc, medP, color=colP, lw=2.6, alpha=0.95, zorder=3, label="Median SNR P (binned)")
    ax.plot(xc, medS, color=colS, lw=2.6, alpha=0.95, zorder=3, label="Median SNR S (binned)")

    # Scatter points
    ax.scatter(x, snrP, s=point_size, c=colP, alpha=0.35, edgecolors="none",
               rasterized=True, label="P observations", zorder=2)
    ax.scatter(x, snrS, s=point_size, c=colS, alpha=0.35, edgecolors="none",
               rasterized=True, label="S observations", zorder=2)

    # Log y with clean ticks
    ax.set_yscale("log")
    ax.yaxis.set_major_locator(LogLocator(base=10.0, numticks=6))
    ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs=tuple(range(2, 10)), numticks=12))
    ax.yaxis.set_minor_formatter(NullFormatter())

    # Labels / limits
    pad = 0.02 * (xmax - xmin if xmax > xmin else 1.0)
    ax.set_xlim(xmin - pad, xmax + pad)
    ax.set_xlabel("Slope (degrees)")
    ax.set_ylabel("SNR = RMS_signal / RMS_noise")
    if title:
        ax.set_title(title, pad=10)

    # Legend — concise and readable
    handles = [
        Line2D([0], [0], marker='o', color='none', markerfacecolor=colP, alpha=0.55,
               markersize=7, label="P observations"),
        Line2D([0], [0], marker='o', color='none', markerfacecolor=colS, alpha=0.55,
               markersize=7, label="S observations"),
        Line2D([0], [0], color=colP, lw=2.6, label="Median P (binned)"),
        Line2D([0], [0], color=colS, lw=2.6, label="Median S (binned)"),
    ]
    if n_boot:
        handles.append(Patch(facecolor="0.3", alpha=0.12, edgecolor="none", label="95% CI (bootstrap)"))
    ax.legend(handles=handles, loc="lower right", frameon=True, framealpha=0.1)

    # Grid and layout
    ax.grid(True, which="major", linestyle="--", linewidth=0.8, alpha=0.35)
    fig.tight_layout()

    return fig, ax


In [ ]:
fig, ax = plot_snr_vs_slope(
    dfslope, rms_P, rms_S, rms_N,
    chan_slice=(0, 3190),
    point_size=10,
    smooth_bins=20,
    n_boot=200,                 # set 0 to hide CI bands
    title="SNR vs slope by channel"
)
plt.show()


In [ ]:
P_cor_all = rms_P * R_used
S_cor_all = rms_S * R_used
plot_rms_vs_slope(dfslope, P_cor_all, S_cor_all, chan_slice=(0, 3190), point_size=10, smooth_bins=20)

## Compare S-wave amplitudes by lithology
Inspect channel counts and corrected-amplitude distributions. Interpret spatial association cautiously: lithology, slope, coupling and orientation can covary. Correct legacy axis labels before publication.


In [ ]:
#improved version used chatgpt suggestions
# Pretty / professional violin plot of S_amplitude by Lithology
# --- Inputs expected ---
import textwrap
from matplotlib.ticker import LogLocator, NullFormatter
assert "Lithology" in dfslope.columns, "dfslope must have a 'Lithology' column."
assert len(S_cor_all) == len(dfslope), "S_cor_all length must match dfslope."

# --- Colors + mapping (kept from your code) ---
lithology_colors_real = {
    'Superficial Deposits': '#FFD700',
    'Andesite': '#DC143C',
    'Volcanic Breccia': '#8B4513',
    'Granodiorite': '#FF8C00',
    'Diorite': '#228B22',
    'Diabase, Basalt': '#9370DB'
}
lithology_names = {
    'Qs':  'Superficial Deposits',
    'Qra': 'Andesite',
    'To':  'Volcanic Breccia',
    'Tg':  'Granodiorite',
    'Tdi': 'Diorite',
    'Td':  'Diabase, Basalt'
}

# --- Prep data ---
dfv = dfslope.copy()
dfv["S_amplitude"]    = np.asarray(S_cor_all, float)
dfv["Lithology_name"] = (
    dfv["Lithology"].map(lithology_names).fillna(dfv["Lithology"]).astype(str).str.strip()
)

# Keep only positive amplitudes (log10 needs > 0)
dfv = dfv[np.isfinite(dfv["S_amplitude"]) & (dfv["S_amplitude"] > 0)].copy()
dfv["log10_S"] = np.log10(dfv["S_amplitude"])

# Order & palette (fixed so colors don’t shift)
order = [c for c in ["Andesite","Granodiorite","Superficial Deposits",
                     "Volcanic Breccia","Diorite","Diabase, Basalt"]
         if c in dfv["Lithology_name"].unique()]
palette = {k: lithology_colors_real.get(k, "#AAAAAA") for k in order}

# --- Group stats for medians / counts ---
g = dfv.groupby("Lithology_name", sort=False)["log10_S"]
med_log = g.median()
counts  = g.size()

# --- Theme ---
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.dpi": 140, "savefig.dpi": 300,
    "axes.linewidth": 1.2,
    "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": "0.85", "grid.linestyle": "--", "grid.linewidth": 0.8,
    "axes.labelpad": 10, "xtick.major.pad": 6, "ytick.major.pad": 6,
})

fig, ax = plt.subplots(figsize=(12.5, 6.5))

# --- Violin on log10 values (no log scaling artifacts) ---
sns.violinplot(
    x="Lithology_name", y="log10_S",
    data=dfv, order=order, palette=palette,
    inner="quartile", cut=0, scale="width",
    bw_adjust=0.7, linewidth=1.2, ax=ax
)

# --- Jittered observations (in log10) ---
sns.stripplot(
    x="Lithology_name", y="log10_S",
    data=dfv, order=order,
    color="black", alpha=0.30, jitter=True, size=3, linewidth=0, ax=ax, zorder=2
)

# --- Median diamonds ---
for i, name in enumerate(order):
    if name in med_log.index:
        ymed = med_log.loc[name]
        ax.scatter(i, ymed, marker="D", s=55, color="black",
                   edgecolor="white", linewidth=0.6, zorder=3)

# --- Y-axis: label in powers of 10 ---
ymin = np.floor(dfv["log10_S"].min())
ymax = np.ceil(dfv["log10_S"].max())
ax.set_ylim(ymin - 0.1, ymax + 0.1)
ax.yaxis.set_major_locator(FixedLocator(np.arange(ymin, ymax + 1)))
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: rf"$10^{{{int(v)}}}$"))

# --- Labels, title, and x-ticks with sample sizes ---
ax.set_xlabel("Lithology", labelpad=8)
ax.set_ylabel(r"$\log_{10}(\mathrm{Strain\ rate}\ [1/s])$", labelpad=8)

labels_with_n = [f"{textwrap.fill(name, 18)}\n(n={int(counts.get(name, 0))})" for name in order]
ax.set_xticklabels(labels_with_n, rotation=0, ha="center")

#ax.set_title("Distribution of S amplitude by lithology (log10 space)", pad=12)

# --- Legend explaining marks ---
legend_elements = [
    Patch(facecolor="#cccccc", edgecolor="black", label="Violin (with quartiles)"),
    Line2D([0], [0], marker="D", color="none", markerfacecolor="black",
           markeredgecolor="white", markersize=7, label="Median"),
    Line2D([0], [0], marker="o", color="none", markerfacecolor="black",
           alpha=0.30, markersize=6, label="Observation")
]
ax.legend(handles=legend_elements, loc="upper right", frameon=True, framealpha=0.1)

fig.tight_layout()
fig.subplots_adjust(bottom=0.22)  # space for two-line xticklabels
plt.savefig('violin-lithology.png',dpi=300)
plt.show()


## Compare cable orientation and map layers
Compute geometry-dependent orientation factors and inspect their relation to amplitudes. The remaining cells assemble exploratory correlations and layered maps; verify channel alignment and units throughout.


In [ ]:
# --- Same CRS + projection as before ---
crs_wgs84 = CRS("EPSG:4326")     # WGS84 latitude and longitude
crs_utm10n = CRS("EPSG:32610")   # UTM zone 10N
transformer = Transformer.from_crs(crs_wgs84, crs_utm10n, always_xy=True)

# Convert longitude and latitude to x and y in meters -> kilometers
x_meters, y_meters = transformer.transform(df['Longitude [°]'].values, df['Latitude [°]'].values)
df['x'] = x_meters / 1000.0
df['y'] = y_meters / 1000.0

# Earthquake coordinates (lon, lat) -> kilometers
quake_lat = 46.7722
quake_lon = -121.9588
quake_x_meters, quake_y_meters = transformer.transform(quake_lon, quake_lat)  # (lon, lat) order
quake_x = quake_x_meters / 1000.0
quake_y = quake_y_meters / 1000.0

# --- Remove midpoint code entirely ---
# (No lat_midpoints/lon_midpoints/x_mid/y_mid used)

# --- Helper (same as yours, but we won’t need it for quake->channel) ---
def compute_theta(x1, y1, x2, y2):
    return np.arctan2(y2 - y1, x2 - x1)

# --- Per-channel fiber orientation using central differences (constant spacing 9.7 m = 0.0097 km) ---
# Option A (clean & robust): gradient with physical spacing
dx = np.gradient(df['x'].to_numpy(), 0.0097)
dy = np.gradient(df['y'].to_numpy(), 0.0097)
theta_fiber = np.arctan2(dy, dx)                     # length N (one angle per channel)

# (If you prefer the explicit central-difference form, use the three lines below instead of np.gradient)
# dx = np.empty_like(df['x'].to_numpy()); dy = np.empty_like(df['y'].to_numpy())
# dx[1:-1] = 0.5 * (df['x'].to_numpy()[2:] - df['x'].to_numpy()[:-2]);  dx[0]  = df['x'].to_numpy()[1]  - df['x'].to_numpy()[0];  dx[-1] = df['x'].to_numpy()[-1] - df['x'].to_numpy()[-2]
# dy[1:-1] = 0.5 * (df['y'].to_numpy()[2:] - df['y'].to_numpy()[:-2]);  dy[0]  = df['y'].to_numpy()[1]  - df['y'].to_numpy()[0];  dy[-1] = df['y'].to_numpy()[-1] - df['y'].to_numpy()[-2]
# theta_fiber = np.arctan2(dy, dx)

# --- Per-channel direction from earthquake to each channel ---
theta_quake = np.arctan2(df['y'].to_numpy() - quake_y, df['x'].to_numpy() - quake_x)   # length N

# --- Angle difference and wrapping to (-pi, pi] ---
theta_diff = (theta_quake - theta_fiber + np.pi) % (2 * np.pi) - np.pi

# --- Orientation components (Kennett 3.12–3.13) ---
radial_component     = np.cos(theta_diff) ** 2              # in [0, 1]
tangential_component = 0.5 * np.sin(2 * theta_diff)         # signed, in [-0.5, 0.5]

# --- Safe normalization for plotting (optional) ---
def minmax01(a):
    a = np.asarray(a, float)
    amin, amax = np.nanmin(a), np.nanmax(a)
    rng = amax - amin
    if not np.isfinite(amin) or not np.isfinite(amax) or rng == 0:
        return np.zeros_like(a)
    return (a - amin) / rng

radial_normalized     = minmax01(radial_component)
tangential_normalized = minmax01(tangential_component)   # note: normalization removes sign; keep raw if sign matters


In [ ]:

from scipy.stats import pearsonr, spearmanr
F_rad = radial_normalized
F_tan = tangential_normalized
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, NullFormatter

# --- Inputs expected (already computed) ---
# P_cor_all, S_cor_all : arrays (len N), >0
# F_rad, F_tan         : arrays (len N); F_rad in [0,1]; F_tan can be signed or normalized

# If you normalized tangential and want sign, swap this line for your raw F_tan:
# F_tan = 0.5 * np.sin(2 * psi)

# Define the 4 plot pairs (note: use \frac not \tfrac)
pairs = [
    (r"$F_{\mathrm{rad}}=\cos^2\psi$",           F_rad, P_cor_all, "P$_{cor}$", "#d62728"),  # red
    (r"$F_{\mathrm{rad}}=\cos^2\psi$",           F_rad, S_cor_all, "S$_{cor}$", "#1f77b4"),  # blue
    (r"$F_{\mathrm{tan}}=\frac{1}{2}\sin(2\psi)$", F_tan, P_cor_all, "P$_{cor}$", "#d62728"),
    (r"$F_{\mathrm{tan}}=\frac{1}{2}\sin(2\psi)$", F_tan, S_cor_all, "S$_{cor}$", "#1f77b4"),
]

fig, axes = plt.subplots(2, 2, figsize=(12.5, 8.5), sharey=True)

for ax, (xlabel, xvals, yvals, lab, col) in zip(axes.flat, pairs):
    # finite + positive y for log scale
    m = np.isfinite(xvals) & np.isfinite(yvals) & (yvals > 0)
    x = np.asarray(xvals)[m]
    y = np.asarray(yvals)[m]

    ax.scatter(x, y, s=14, alpha=0.35, color=col, edgecolors="none", label=lab, rasterized=True)

    ax.set_yscale("log")
    ax.yaxis.set_major_locator(LogLocator(base=10.0, numticks=6))
    ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs=tuple(range(2,10)), numticks=12))
    ax.yaxis.set_minor_formatter(NullFormatter())

    if x.size >= 2:
        xmin, xmax = float(np.nanmin(x)), float(np.nanmax(x))
        pad = 0.02 * (xmax - xmin if xmax > xmin else 1.0)
        ax.set_xlim(xmin - pad, xmax + pad)

    ax.set_xlabel(xlabel)
    ax.grid(True, which="major", linestyle="--", linewidth=0.8, alpha=0.35)
    ax.legend(loc="lower right", frameon=True, framealpha=0.1)

axes[0,0].set_ylabel("Amplitude (P$_{cor}$/S$_{cor}$)")
axes[1,0].set_ylabel("Amplitude (P$_{cor}$/S$_{cor}$)")

fig.suptitle("R-corrected amplitudes vs orientation factors", y=0.995)
fig.tight_layout()
plt.show()

In [ ]:
df['F_rad']     = F_rad
df['F_tan']     = F_tan
df['P_cor_all'] = P_cor_all
df['S_cor_all'] = S_cor_all
df['Noise']     = rms_N   # or whatever your noise per channel is

In [ ]:
# 1) Make a clean lithology code column and handle NaNs
df['LithoCode'] = df['Lithology'].astype('string').str.strip()
df['LithoCode'] = df['LithoCode'].fillna('Unknown')

# 2) Map codes -> pretty names (optional; use if your labels/color dict are by name)
code_to_name = {
    'Qs':  'Superficial Deposits',
    'Qra': 'Andesite',
    'To':  'Volcanic Breccia',
    'Tg':  'Granodiorite',
    'Tdi': 'Diorite',
    'Td':  'Diabase, Basalt',
    'Unknown': 'Unknown'
}
df['Lithology_name'] = df['LithoCode'].map(code_to_name).fillna('Unknown')

# 3) Colors – key them by the same field you’ll iterate on.
#    (A) If you want to color by names:
lithology_colors_by_name = {
    'Superficial Deposits': '#FFD700',
    'Andesite': '#DC143C',
    'Volcanic Breccia': '#8B4513',
    'Granodiorite': '#FF8C00',
    'Diorite': '#228B22',
    'Diabase, Basalt': '#9370DB',
    'Unknown': '#A9A9A9'  # grey
}
#    (B) Or, if you prefer to color directly by codes, use this instead:
lithology_colors_by_code = {
    'Qs':  '#FFD700',
    'Qra': '#DC143C',
    'To':  '#8B4513',
    'Tg':  '#FF8C00',
    'Tdi': '#228B22',
    'Td':  '#9370DB',
    'Unknown': '#A9A9A9'
}

lithology_colors_by_code['Tdi'] = '#228B22'      # green
lithology_colors_by_name['Diorite'] = '#228B22'  # if you ever use names


In [ ]:
# BEFORE plotting
df['Slope_deg'] = np.asarray(dfslope['slope_deg'], float)

# then in plotting:
slope_vals = df['slope_deg'].to_numpy()


In [ ]:
# Helper: split a (x,y)-ordered DataFrame into contiguous segments
# threshold is a gap in km (e.g., 0.1 km = 100 m)
def split_into_segments(df_seg, threshold=0.1):
    df_seg = df_seg.sort_values(["x","y"]).reset_index(drop=True)  # or sort by Channel if you have it
    if threshold is None or threshold <= 0 or len(df_seg) < 2:
        return [df_seg.copy()]

    x = df_seg["x"].to_numpy()
    y = df_seg["y"].to_numpy()

    segs = []
    start = 0
    for i in range(1, len(df_seg)):
        if np.hypot(x[i] - x[i-1], y[i] - y[i-1]) > threshold:
            segs.append(df_seg.iloc[start:i].copy())
            start = i
    segs.append(df_seg.iloc[start:].copy())
    return [s for s in segs if len(s) > 0]


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colorbar import ColorbarBase
from matplotlib import patheffects as pe

# -------------------- Pretty style tweaks (layout unchanged) --------------------
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300,
    "font.size": 12,
    "axes.titlesize": 16, "axes.labelsize": 13,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.linestyle": "--", "grid.alpha": 0.12,
})
# Ensure the codes in the DataFrame have no stray spaces and match the dict keys
df['Lithology'] = df['Lithology'].astype('string').str.strip()

# Make Tdi (Diorite) green
lithology_colors.update({'Tdi': '#228B22'})   # ForestGreen
# -------------------- 1) Layers + offsets (unchanged) --------------------
layer_order = ['Lithology', 'Radial', 'Tangential', 'Slope', 'P_cor_all', 'S_cor_all']
base_offset = 0.0
spacing     = 2.2
offsets     = {layer: base_offset + i*spacing for i, layer in enumerate(layer_order)}

label_names = {
    'Lithology' : 'Lithology',
    'Radial'    : r'Radial factor ',
    'Tangential': r'Tangential factor',
    'Slope'     : 'Slope (deg)',
    'P_cor_all' : r'P amplitude ',
    'S_cor_all' : r'S amplitude ',
}
label_shift = 0.55

# -------------------- 2) Figure (unchanged geometry) --------------------
fig, ax = plt.subplots(figsize=(17, 14))
plt.subplots_adjust(left=0.08, right=0.95, bottom=0.07, top=0.94)

xmin, xmax = float(np.nanmin(df['x'])), float(np.nanmax(df['x']))
for lname in layer_order:
    yline = float(np.nanmin(df['y'])) + offsets[lname]
    ax.plot([xmin-0.5, xmax+0.5], [yline-0.15, yline-0.15], color='k', lw=0.4, alpha=0.15)

# -------------------- 3) Lithology track (respect your colors) --------------------
df['Lithology'] = df['Lithology'].astype('string').str.strip().fillna('Unknown')

# ... dentro de tu bloque de “Lithology track”:
for lithology, color in lithology_colors.items():
    sub = df[df['Lithology'] == lithology][['x','y']].dropna()
    if sub.empty:
        continue
    try:
        segments = split_into_segments(sub, threshold=0.3)  # un poco más laxo que 0.1 km
        for seg in segments:
            # <--- QUITA el filtro len(seg) > 1 para NO perder singletons
            ax.scatter(seg['x'], seg['y'] + offsets['Lithology'],
                       color=color, s=42, alpha=0.95, linewidth=0.10, edgecolors='k')
    except NameError:
        # fallback: plotea todo sin segmentar
        ax.scatter(sub['x'], sub['y'] + offsets['Lithology'],
                   color=color, s=42, alpha=0.95, linewidth=0.10, edgecolors='k')

# -------------------- 4) Colormaps + norms (RESPECT your cmaps) --------------------
# F_rad in [0,1]
rad_norm  = mcolors.Normalize(vmin=0.0, vmax=1.0)
rad_cmap  = 'Spectral_r'   # your cmap

# F_tan (signed)
tan_vals_all = np.asarray(df['F_tan'], float)
tan_vals = tan_vals_all[np.isfinite(tan_vals_all)]
if tan_vals.size == 0:
    tan_norm = mcolors.Normalize(vmin=-0.5, vmax=0.5)
    tan_cmap = 'Spectral_r'
else:
    p1, p99 = np.nanpercentile(tan_vals, [1, 99])
    if p1 > p99:
        p1, p99 = p99, p1
    if (p1 < 0.0) and (p99 > 0.0):
        vmax_abs = max(abs(p1), abs(p99))
        tan_norm = mcolors.TwoSlopeNorm(vmin=-vmax_abs, vcenter=0.0, vmax=vmax_abs)
        tan_cmap = 'viridis'      # as in your earlier snippet
    else:
        vmin = float(np.nanmin(tan_vals)); vmax = float(np.nanmax(tan_vals))
        if vmin == vmax:
            vmin -= 1e-6; vmax += 1e-6
        tan_norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
        tan_cmap = 'Spectral_r'

# slope (deg)
if 'slope_deg' not in df.columns:
    raise KeyError("df['slope_deg'] not found. Merge dfslope['slope_deg'] into df first.")
slope_vals = np.asarray(df['slope_deg'], float)
s_v = np.nanpercentile(slope_vals, [0.5, 99.5])
slope_norm = mcolors.Normalize(vmin=s_v[0], vmax=s_v[1])
slope_cmap = 'Spectral_r'

# amplitudes -> log10 with robust [2,98] (cmap unchanged)
P_vals = np.asarray(df['P_cor_all'], float)
S_vals = np.asarray(df['S_cor_all'], float)
P_log  = np.log10(np.clip(P_vals, 1e-15, None))
S_log  = np.log10(np.clip(S_vals, 1e-15, None))

P_fin = P_log[np.isfinite(P_log)]
S_fin = S_log[np.isfinite(S_log)]
if P_fin.size == 0: P_fin = np.array([0.0, 1.0])
if S_fin.size == 0: S_fin = np.array([0.0, 1.0])

P_lo, P_hi = np.nanpercentile(P_fin, [2, 98])
S_lo, S_hi = np.nanpercentile(S_fin, [2, 98])
if P_lo == P_hi: P_lo, P_hi = P_lo - 0.1, P_hi + 0.1
if S_lo == S_hi: S_lo, S_hi = S_lo - 0.1, S_hi + 0.1

P_norm   = mcolors.Normalize(vmin=P_lo, vmax=P_hi)
S_norm   = mcolors.Normalize(vmin=S_lo, vmax=S_hi)
amp_cmap = 'Spectral_r'

# -------------------- 5) Scatter layers (LIGHTER: alpha=0.80, edges white) --------------------
im_radial = ax.scatter(
    df['x'], df['y'] + offsets['Radial'],
    c=df['F_rad'], cmap=rad_cmap, norm=rad_norm,
    s=42, alpha=0.80, edgecolor='k', linewidth=0.10
)

im_tangential = ax.scatter(
    df['x'], df['y'] + offsets['Tangential'],
    c=tan_vals_all, cmap=tan_cmap, norm=tan_norm,
    s=42, alpha=0.80, edgecolor='k', linewidth=0.10
)

im_slope = ax.scatter(
    df['x'], df['y'] + offsets['Slope'],
    c=slope_vals, cmap=slope_cmap, norm=slope_norm,
    s=42, alpha=0.80, edgecolor='k', linewidth=0.10
)

# P/S colored by log10 amplitude (same cmap/layout)
im_p = ax.scatter(
    df['x'], df['y'] + offsets['P_cor_all'],
    c=P_log, cmap=amp_cmap, norm=P_norm,
    s=42, alpha=0.80, edgecolor='k', linewidth=0.10
)

im_s = ax.scatter(
    df['x'], df['y'] + offsets['S_cor_all'],
    c=S_log, cmap=amp_cmap, norm=S_norm,
    s=42, alpha=0.80, edgecolor='k', linewidth=0.10
)

# -------------------- 6) Axes cleanup (unchanged) --------------------
ax.set_xticks([]); ax.set_yticks([])

# -------------------- 7) Scale bar + North arrow (unchanged positions) --------------------
xlims = ax.get_xlim(); ylims = ax.get_ylim()
x_start = xlims[0] - 1.2; y_start = ylims[0] - 1.2

scalebar_length = 5  # km
ax.plot([x_start, x_start + scalebar_length], [y_start, y_start], color='k', lw=3, solid_capstyle='butt')
ax.text(x_start + scalebar_length/2, y_start - 0.03*(ylims[1]-ylims[0]),
        f'{scalebar_length} km', ha='center', va='top', fontsize=12,
        path_effects=[pe.withStroke(linewidth=2, foreground='white', alpha=0.8)])

arrow_len = 1
ax.annotate('', xy=(x_start, y_start + arrow_len), xytext=(x_start, y_start),
            arrowprops=dict(facecolor='k', width=4, headwidth=12, shrinkA=0, shrinkB=0))
ax.text(x_start, y_start + arrow_len + 0.03*(ylims[1]-ylims[0]),
        'N', ha='center', va='bottom', fontsize=12,
        path_effects=[pe.withStroke(linewidth=2, foreground='k', alpha=0.8)])

# -------------------- 8) Earthquake marker (unchanged) --------------------
ax.scatter(quake_x, quake_y, color='gold', s=180, marker='*',
           edgecolor='black', linewidth=1.2, zorder=5)
ax.text(quake_x + 0.2, quake_y + 0.2, 'Ml 2.9, Aug 27, 2023',
        fontsize=12, color='black', fontweight='bold', ha='left', va='bottom',
        bbox=dict(facecolor='white', alpha=0.75, edgecolor='0.8'))

# -------------------- 9) Section labels (unchanged positions) --------------------
x_label = xmin - 0.9
y0 = float(np.nanmin(df['y']))
for key, name in label_names.items():
    yy = y0 + offsets[key] + label_shift + 1.0
    ax.text(x_label, yy, name, fontsize=13, ha='left', fontweight='bold', color='black',
            path_effects=[pe.withStroke(linewidth=3, foreground='white', alpha=0.9)])

# -------------------- 10) Lithology colorbar (unchanged rect) --------------------
present = [k for k in lithology_colors if (df['Lithology'] == k).any()]
cmap_litho = mcolors.ListedColormap([lithology_colors[k] for k in present])
norm_litho = mcolors.BoundaryNorm(np.arange(len(present)+1)-0.5, len(present))

cax_litho = ax.inset_axes([0.70, 0.2, 0.27, 0.03])
cb_litho = ColorbarBase(cax_litho, cmap=cmap_litho, norm=norm_litho, orientation='horizontal')
cb_litho.set_ticks(np.arange(len(present)))
cb_litho.ax.set_xticklabels(present, fontsize=11, rotation=20, ha='right')
cb_litho.set_label('Lithology', fontsize=13)

# -------------------- 11) Colorbars (same positions as before) --------------------
def add_hbar(mappable, rect, label):
    cax = ax.inset_axes(rect)
    cb  = plt.colorbar(mappable, cax=cax, orientation='horizontal')
    cb.set_label(label, fontsize=12)
    cb.ax.tick_params(labelsize=10)
    return cb

add_hbar(im_radial,    [0.08, 1, 0.25, 0.03], 'F_rad')
add_hbar(im_tangential,[0.40, 1, 0.25, 0.03], 'F_tan ')
add_hbar(im_slope,     [0.72, 1, 0.25, 0.03], 'Slope (deg)')
add_hbar(im_p,         [0.72, 0.1,  0.25, 0.03], r'log$_{10}$ P amplitude')
add_hbar(im_s,         [0.4, 0.1,  0.25, 0.03], r'log$_{10}$ S amplitude')

# -------------------- 12) Save/show --------------------
plt.savefig('cable_layers_log_amp_lighter.png', dpi=300, bbox_inches='tight')
plt.show()
